<a href="https://colab.research.google.com/github/sandamali-2002/Production-Ready-RAG-Application-Ask-My-Docs/blob/main/Hybrid_Search_RAG_using_Langchain_and_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf -q
!pip install langchain -q
!pip install langchain-openai -q
!pip install langchain_chroma -q
!pip install langchain_community -q

!pip install rank_bm25 -q



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71

In [2]:
import os
from google.colab import userdata

Initialize OpenAI LLM

In [3]:
from langchain_openai import ChatOpenAI
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
llm=ChatOpenAI(
    model='gpt-3.5-turbo',
    temperature=0.1,
)



Initialize Embedding Model

In [4]:
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings(model='text-embedding-3-small')

Load PDF Document

In [8]:
from langchain_community.document_loaders import PyPDFLoader
loder=PyPDFLoader("/content/codeprolk.pdf")
docs=loder.load()

Split Documents into Chunks

In [13]:
!pip install langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=250,chunk_overlap=30)

chunks = splitter.split_documents(docs)



In [14]:
len(chunks)

33

Create Semantic Search Retriever

In [15]:
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(chunks, embedding_model)

vectorstore_retreiver = vectorstore.as_retriever(search_kwargs={"k": 2})


Create Keyword Search Retriever

In [17]:
from langchain_community.retrievers import BM25Retriever

keyword_retriever = BM25Retriever.from_documents(chunks)

keyword_retriever.k =  2

Create Hybrid Search Retriever

In [18]:
!pip install langchain-classic

from langchain_classic.retrievers.ensemble import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[vectorstore_retreiver, keyword_retriever],  # Fixed typo
    weights=[0.5, 0.5]
)


Define Prompt Template

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt=ChatPromptTemplate.from_messages(["human",message])

Create RAG Chain with Hybrid Search

In [21]:
chain=(
    {
        "context": ensemble_retriever,
        "question":RunnablePassthrough()
    }

    | prompt
    | llm
)

Invoke RAG Chain with Example Questions

In [22]:
response = chain.invoke("what are the popular videos in codeprolk")

print(response.content)

The popular videos in CodePRO LK are those that have assisted viewers in their learning journeys and have played a significant role in democratizing tech.


Evaluate

In [23]:
import pandas as pd
test_data=pd.read_csv("/content/test_data (1).csv")
test_data


,question,answer
0,what is codeprolk,CodePRO LK is a dynamic educational platform ...
1,what is the main vision of them,To assist talented Sri Lankans in reaching th...
2,what is the mission of codeprolk,To produce high-quality tech courses and arti...
3,What are the courses they offer,CodeProLK offers courses like Python GUI with...
4,These courses are for which nationality,CodeProLK's courses are designed specifically...
5,what are the popular videos in codeprolk youtu...,It features popular videos like Python Basics...


In [24]:
question=test_data["question"].to_list()
ground_truth=test_data["answer"].to_list()
#

In [25]:
data={"question":[],"answer":[],"context":[],"ground_truth":[]}